# Data Analysis Pipeline: Modelado Descriptivo

**BCN-Meteorologics** - Accidentes de tráfico en Barcelona (2025) con datos meteorológicos

## Descripción del Pipeline

El segundo Data Analysis Pipeline implementado es un sistema de **visualización descriptiva e interactiva** que opera directamente sobre la tabla `T_UNIFIED` dentro de la Exploitation Zone. A diferencia del pipeline predictivo, cuyo objetivo es estimar P(Y | X, M) a nivel de instancia, este flujo tiene como propósito construir una **comprensión estructural del fenómeno de la siniestralidad vial en Barcelona**: dónde se concentra, cuándo ocurre, bajo qué condiciones meteorológicas y a qué perfiles demográficos afecta con mayor severidad.

El pipeline se ha desarrollado en este notebook apoyándose en dos tecnologías complementarias:
- **Folium** — cartografía interactiva multicapa
- **Plotly** — gráficos estadísticos dinámicos

El dataset de trabajo consolida la información del año 2025 en el municipio de Barcelona, abarcando **7.741 accidentes únicos** (tabla `T_ACCIDENTS`) y **16.001 registros** a nivel de persona-accidente (tabla `T_UNIFIED`).

---

**Contenido del notebook:**

| Sección | Descripción |
|---------|-------------|
| 1. Setup | Carga de librerías y datos desde DuckDB |
| 2. Variables derivadas | Clasificación de gravedad |
| 3. Mapa interactivo principal | Heatmap, marcadores, distritos y estaciones |
| 4. Mapa por barrio | Scatter geográfico de siniestralidad |
| 5. Análisis temporal | Distribución mensual, semanal y horaria |
| 6. Condiciones meteorológicas | Temperatura, humedad y gravedad |
| 7. Análisis por distrito | Stacked bar de gravedad por distrito |
| 8. Perfil de riesgo | Vehículo y demografía |
| 9. Mapa animado | Evolución mensual |
| 10. Heatmap hora-día | Momentos de mayor riesgo |
| 11. Exportación | HTML interactivo |

## 1. Setup y carga de datos

In [13]:
import sys
!{sys.executable} -m pip install folium plotly duckdb nbformat --quiet


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: C:\Users\jche2\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [14]:
from pathlib import Path

import duckdb
import pandas as pd
import folium
from folium.plugins import HeatMap, MarkerCluster
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

db_path = Path(__file__).resolve().parent / "exploit_zone.db" if "__file__" in dir() else Path("exploit_zone.db")
if not db_path.exists():
    db_path = Path(r"c:\UPC\BCN-Meteorologics\exploit_zone\exploit_zone.db")

con = duckdb.connect(str(db_path), read_only=True)

df_unified = con.execute("SELECT * FROM T_UNIFIED").fetchdf()
df_meteo = con.execute("SELECT * FROM T_METEO").fetchdf()
df_accidents = con.execute("SELECT * FROM T_ACCIDENTS").fetchdf()

con.close()

print(f"T_UNIFIED: {df_unified.shape[0]} filas, {df_unified.shape[1]} columnas")
print(f"T_METEO:   {df_meteo.shape[0]} filas")
print(f"T_ACCIDENTS: {df_accidents.shape[0]} filas")
df_unified.head(3)

T_UNIFIED: 16001 filas, 52 columnas
T_METEO:   1095 filas
T_ACCIDENTS: 7741 filas


,numero_expedient,codi_districte,nom_districte,codi_barri,nom_barri,codi_carrer,nom_carrer,num_postal,descripcio_dia_setmana,nk_any,...,PPT,PX,RS24h,TM,TN,TX,VVM10,VVX10,meteo_latitud_wgs84,meteo_longitud_wgs84
0,2025S002330,2,Eixample,8,l'Antiga Esquerra de l'Eixample,191204,Mallorca,152,Dimarts,2025,...,0.0,1018.7,27.3,16.9,14.7,19.5,2.2,6.5,41.3839,2.16775
1,2025S002842,9,Sant Andreu,59,el Bon Pastor,701371,República Dominicana,1-23,Dijous,2025,...,1.3,1013.5,17.2,18.1,15.7,20.4,2.3,8.1,41.3839,2.16775
2,2025S002909,3,Sants-Montjuïc,18,Sants,296406,Sant Antoni,55,Diumenge,2025,...,0.0,1016.5,29.8,21.0,17.4,24.5,1.6,6.4,41.3839,2.16775


## 2. Preparación de variables derivadas

Creamos columnas auxiliares que serán útiles para la visualización:
- `gravedad`: categoría de gravedad del accidente (Mortal / Grave / Leve / Sin víctimas)
- `color_gravedad`: color asignado a cada categoría para los marcadores del mapa

In [15]:
# Clasificación de gravedad
def classify_severity(row):
    if row["numero_morts"] > 0:
        return "Mortal"
    if row["numero_lesionats_greus"] > 0:
        return "Grave"
    if row["numero_lesionats_lleus"] > 0:
        return "Leve"
    return "Sin víctimas"

df_unified["gravedad"] = df_unified.apply(classify_severity, axis=1)

SEVERITY_COLORS = {
    "Mortal": "black",
    "Grave": "red",
    "Leve": "orange",
    "Sin víctimas": "blue",
}
df_unified["color_gravedad"] = df_unified["gravedad"].map(SEVERITY_COLORS)

# Filtrar filas con coordenadas válidas
df_geo = df_unified.dropna(subset=["latitud_wgs84", "longitud_wgs84"]).copy()

print(f"Registros con coordenadas: {len(df_geo)} / {len(df_unified)}")
print(f"\nDistribución de gravedad:")
print(df_geo["gravedad"].value_counts())

Registros con coordenadas: 16001 / 16001

Distribución de gravedad:
gravedad
Leve            14429
Sin víctimas     1005
Grave             545
Mortal             22
Name: count, dtype: int64


### Análisis: Preparación de Variables Derivadas

Como primer paso del pipeline, se ha construido la variable `gravedad`, categorizando cada registro en **cuatro niveles de severidad mutuamente excluyentes y ordenados**:

| Categoría | Criterio | N (Nivel Persona) | N (Nivel Accidente) | % Accidentes |
|-----------|----------|:-----------------:|:-------------------:|:------------:|
| **Mortal** | `numero_morts > 0` | 22 | 11 | 0,14% |
| **Grave** | `numero_lesionats_greus > 0` | 545 | 241 | 3,11% |
| **Leve** | `numero_lesionats_lleus > 0` | 14.429 | 6.599 | 85,26% |
| **Sin víctimas** | Resto | 1.005 | 890 | 11,50% |

La distribución revela un **fuerte desequilibrio de clases**, fuertemente alineado con la literatura sobre seguridad vial urbana: los accidentes con consecuencias letales o graves representan un **3,25% del total**, frente a una abrumadora mayoría de lesiones leves (85,3%).

> **Implicación técnica:** Este desequilibrio justifica el uso del parámetro `scale_pos_weight` en el pipeline predictivo. Desde la perspectiva descriptiva, obliga a emplear escalas logarítmicas o capas visuales independientes para evitar que el volumen de los incidentes leves invisibilice los casos de mayor criticidad.

## 3. Mapa interactivo principal

El mapa integra múltiples capas que se pueden activar/desactivar con el control de capas:

| Capa | Descripción |
|------|-------------|
| **Heatmap de accidentes** | Densidad general de accidentes |
| **Accidentes graves/mortales** | Marcadores individuales de accidentes graves y mortales |
| **Accidentes por distrito** | Círculos proporcionales al número de accidentes por distrito |
| **Estaciones meteorológicas** | Ubicación de las 3 estaciones MeteoCat con sus promedios |

In [16]:
# ---------- Centro del mapa: Barcelona ----------
BCN_CENTER = [41.3920, 2.1640]

m = folium.Map(location=BCN_CENTER, zoom_start=13, tiles="cartodbpositron")

# ========== CAPA 1: Heatmap de densidad ==========
heat_data = df_geo[["latitud_wgs84", "longitud_wgs84"]].values.tolist()
heatmap_layer = HeatMap(
    heat_data,
    name="Heatmap densidad accidentes",
    radius=12,
    blur=15,
    max_zoom=15,
)
heatmap_layer.add_to(m)

# ========== CAPA 2: Accidentes graves y mortales (marcadores individuales) ==========
severe = df_geo[df_geo["gravedad"].isin(["Grave", "Mortal"])]
severe_cluster = MarkerCluster(name="Accidentes graves / mortales")

for _, row in severe.iterrows():
    icon_color = SEVERITY_COLORS[row["gravedad"]]
    popup_html = (
        f"<b>{row['gravedad']}</b><br>"
        f"<b>Fecha:</b> {row['data_accident']}<br>"
        f"<b>Distrito:</b> {row['nom_districte']}<br>"
        f"<b>Barrio:</b> {row['nom_barri']}<br>"
        f"<b>Calle:</b> {row['nom_carrer']}<br>"
        f"<b>Vehículo:</b> {row['desc_tipus_vehicle_implicat']}<br>"
        f"<b>Víctimas:</b> {row['numero_victimes']} | "
        f"Muertos: {row['numero_morts']} | Graves: {row['numero_lesionats_greus']}<br>"
        f"<b>Temp:</b> {row['TM']:.1f}°C | <b>Humedad:</b> {row['HRM']:.0f}% | "
        f"<b>Precip:</b> {row['PPT']:.1f} mm"
    )
    folium.Marker(
        location=[row["latitud_wgs84"], row["longitud_wgs84"]],
        popup=folium.Popup(popup_html, max_width=320),
        icon=folium.Icon(color=icon_color, icon="exclamation-sign", prefix="glyphicon"),
    ).add_to(severe_cluster)

severe_cluster.add_to(m)

# ========== CAPA 3: Agregación por distrito ==========
# Usamos T_ACCIDENTS para no duplicar conteos (T_UNIFIED tiene 1 fila por persona)
district_stats = (
    df_accidents.groupby("nom_districte")
    .agg(
        total_accidents=("numero_expedient", "nunique"),
        total_morts=("numero_morts", "sum"),
        total_greus=("numero_lesionats_greus", "sum"),
        total_lleus=("numero_lesionats_lleus", "sum"),
        lat=("latitud_wgs84", "mean"),
        lon=("longitud_wgs84", "mean"),
    )
    .reset_index()
)

district_layer = folium.FeatureGroup(name="Accidentes por distrito")
max_acc = district_stats["total_accidents"].max()

for _, row in district_stats.iterrows():
    if pd.isna(row["lat"]):
        continue
    radius = 8 + (row["total_accidents"] / max_acc) * 35
    popup_html = (
        f"<b>{row['nom_districte']}</b><br>"
        f"Accidentes: {row['total_accidents']}<br>"
        f"Muertos: {int(row['total_morts'])} | Graves: {int(row['total_greus'])} | Leves: {int(row['total_lleus'])}"
    )
    folium.CircleMarker(
        location=[row["lat"], row["lon"]],
        radius=radius,
        color="#e74c3c",
        fill=True,
        fill_color="#e74c3c",
        fill_opacity=0.5,
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=row["nom_districte"],
    ).add_to(district_layer)

district_layer.add_to(m)

# ========== CAPA 4: Estaciones meteorológicas ==========
meteo_stations = df_meteo.groupby("codi_estacio").agg(
    lat=("latitud_wgs84", "first"),
    lon=("longitud_wgs84", "first"),
    temp_media=("TM", "mean"),
    humedad_media=("HRM", "mean"),
    precip_total=("PPT", "sum"),
    viento_medio=("VVM10", "mean"),
).reset_index()

meteo_layer = folium.FeatureGroup(name="Estaciones meteorológicas")

for _, st in meteo_stations.iterrows():
    if pd.isna(st["lat"]):
        continue
    popup_html = (
        f"<b>Estación {st['codi_estacio']}</b><br>"
        f"<b>Temp media:</b> {st['temp_media']:.1f} °C<br>"
        f"<b>Humedad media:</b> {st['humedad_media']:.1f} %<br>"
        f"<b>Precipitación acumulada:</b> {st['precip_total']:.1f} mm<br>"
        f"<b>Viento medio:</b> {st['viento_medio']:.1f} m/s"
    )
    folium.Marker(
        location=[st["lat"], st["lon"]],
        popup=folium.Popup(popup_html, max_width=280),
        tooltip=f"Estación {st['codi_estacio']}",
        icon=folium.Icon(color="green", icon="cloud", prefix="glyphicon"),
    ).add_to(meteo_layer)

meteo_layer.add_to(m)

# ========== Leyenda ==========
legend_html = """
<div style="position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px; border-radius:8px;
     border:2px solid grey; font-size:13px; line-height:1.6;">
<b>Leyenda</b><br>
&#9899; Accidente mortal<br>
<span style="color:red;">&#9899;</span> Accidente grave<br>
<span style="color:orange;">&#9899;</span> Accidente leve<br>
<span style="color:green;">&#9899;</span> Estación meteorológica
</div>
"""
m.get_root().html.add_child(folium.Element(legend_html))

# Control de capas
folium.LayerControl(collapsed=False).add_to(m)

m

### Análisis: Visualización 1 — Mapa Interactivo Multicapa

Este mapa constituye el artefacto central del pipeline. Construido sobre una proyección **CartoDB Positron** centrada en Barcelona, integra cuatro capas analíticas independientes gestionadas mediante un control dinámico:

**Capa 1 — Heatmap de densidad**

Generado a partir de las 16.001 coordenadas WGS84, este mapa de calor identifica nítidamente las zonas de máxima concentración de incidentes. Los focos de calor (colores rojo-amarillo) trazan el **Eixample** (ejes Gran Via, Aragó y Diagonal), el entorno de Glòries y el trazado de las Rondes, coincidiendo con las arterias de mayor carga vehicular de la ciudad.

**Capa 2 — Marcadores de gravedad máxima**

Renderiza los 567 registros de categorías "Grave" y "Mortal" agrupados dinámicamente mediante `MarkerCluster`. Cada marcador despliega un popup detallado (fecha, ubicación, tipo de vehículo y meteorología). Espacialmente, reiteran la concentración en el Eixample y las grandes vías de penetración urbana (Meridiana, Paral·lel, Diagonal).

**Capa 3 — Círculos proporcionales por distrito**

| Rango | Distrito | Accidentes | Muertos | Graves | Leves |
|:-----:|----------|:----------:|:-------:|:------:|:-----:|
| 1º | Eixample | 1.925 | 1 | 76 | 2.152 |
| 2º | Sant Martí | 1.048 | 2 | 35 | 1.123 |
| 3º | Sants-Montjuïc | 927 | 0 | 28 | 1.028 |
| 4º | Sarrià-Sant Gervasi | 800 | 2 | 27 | 847 |

El Eixample aglutina el **24,9% del total** de la ciudad, doblando al segundo distrito más afectado. Esta asimetría responde a su densa trama ortogonal, su rol como eje conector y la alta penetración de vehículos de movilidad personal. Destaca el caso de **Gràcia**: pese a un menor volumen (417 accidentes), concentra la mayor mortalidad absoluta (3 víctimas), señalándolo como un punto crítico de actuación.

**Capa 4 — Estaciones meteorológicas**

Posiciona las tres estaciones MeteoCat activas en 2025. La estación **D5** (norte, Vall d'Hebron) presenta el perfil térmico más bajo (16,4 °C) y mayor viento (3,89 m/s) por su altitud, mientras que X8 y X4 muestran perfiles litorales y céntricos muy similares (precipitaciones superiores a 690 mm/año).

## 4. Mapa de accidentes por barrio (Choropleth-style)

Visualización con Plotly de la distribución de accidentes por barrio, mostrando el número de accidentes y la gravedad media como mapa de dispersión geográfica (scatter_map).

In [17]:
# Agregación por barrio usando T_ACCIDENTS (1 fila por accidente)
barri_stats = (
    df_accidents.groupby(["nom_districte", "nom_barri"])
    .agg(
        total_accidents=("numero_expedient", "nunique"),
        total_morts=("numero_morts", "sum"),
        total_greus=("numero_lesionats_greus", "sum"),
        total_lleus=("numero_lesionats_lleus", "sum"),
        total_victimes=("numero_victimes", "sum"),
        lat=("latitud_wgs84", "mean"),
        lon=("longitud_wgs84", "mean"),
    )
    .reset_index()
    .dropna(subset=["lat", "lon"])
)

fig_barri = px.scatter_map(
    barri_stats,
    lat="lat",
    lon="lon",
    size="total_accidents",
    color="total_greus",
    color_continuous_scale="YlOrRd",
    hover_name="nom_barri",
    hover_data={
        "nom_districte": True,
        "total_accidents": True,
        "total_morts": True,
        "total_greus": True,
        "total_lleus": True,
        "lat": False,
        "lon": False,
    },
    size_max=40,
    zoom=12,
    center={"lat": 41.392, "lon": 2.164},
    title="Accidentes por barrio - tamaño: total accidentes, color: heridos graves",
)
fig_barri.update_layout(map_style="carto-positron", height=650, margin=dict(l=0, r=0, t=40, b=0))
fig_barri.show()

### Análisis: Visualización 2 — Mapa de Siniestralidad por Barrio

Un gráfico de dispersión espacial (`scatter_map`) dimensiona el **volumen absoluto de accidentes** (tamaño del punto) frente al **recuento de heridos graves** (escala de color) para los 73 barrios de la ciudad.

| Barrio | Distrito | Accidentes | Heridos Graves |
|--------|----------|:----------:|:--------------:|
| la Dreta de l'Eixample | Eixample | 672 | 29 |
| l'Antiga Esquerra de l'Eixample | Eixample | 346 | 12 |
| la Nova Esquerra de l'Eixample | Eixample | 275 | 12 |
| Sant Gervasi - Galvany | Sarrià-Sant Gervasi | 261 | 8 |

La superposición de volumen y gravedad revela **anomalías estadísticas de gran valor**:

- **Les Corts** presenta una tasa de heridos graves sobre accidentes totales del **6,0%** (prácticamente el doble del 3,1% de la media de la ciudad), constituyendo un *"punto negro cualitativo"*.
- Inversamente, la **Marina del Prat Vermell** (221 accidentes, 2,7% graves) dibuja un perfil de siniestralidad de alta frecuencia pero baja intensidad, probablemente vinculado a la logística portuaria.

## 5. Análisis descriptivo - Distribución temporal

Análisis de la distribución de accidentes a lo largo del tiempo: por mes, día de la semana y hora del día.

In [18]:
# Orden lógico de los días de la semana en catalán
DAY_ORDER = ["Dilluns", "Dimarts", "Dimecres", "Dijous", "Divendres", "Dissabte", "Diumenge"]
MONTH_ORDER = ["Gener", "Febrer", "Març", "Abril", "Maig", "Juny",
               "Juliol", "Agost", "Setembre", "Octubre", "Novembre", "Desembre"]

fig_temporal = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Accidentes por mes", "Accidentes por día de la semana", "Accidentes por hora del día"),
)

# Por mes (usamos T_ACCIDENTS para conteo real de accidentes)
month_counts = df_accidents.groupby("nom_mes")["numero_expedient"].nunique().reindex(MONTH_ORDER).dropna()
fig_temporal.add_trace(
    go.Bar(x=month_counts.index, y=month_counts.values, marker_color="#3498db", name="Por mes"),
    row=1, col=1,
)

# Por día de la semana
day_counts = df_accidents.groupby("descripcio_dia_setmana")["numero_expedient"].nunique().reindex(DAY_ORDER).dropna()
fig_temporal.add_trace(
    go.Bar(x=day_counts.index, y=day_counts.values, marker_color="#e74c3c", name="Por día"),
    row=1, col=2,
)

# Por hora
hour_counts = df_accidents.groupby("hora_dia")["numero_expedient"].nunique().reset_index()
hour_counts.columns = ["hora", "total"]
fig_temporal.add_trace(
    go.Scatter(x=hour_counts["hora"], y=hour_counts["total"], mode="lines+markers",
               marker_color="#2ecc71", name="Por hora"),
    row=1, col=3,
)

fig_temporal.update_layout(height=400, showlegend=False, title_text="Distribución temporal de accidentes")
fig_temporal.show()

### Análisis: Visualización 3 — Dinámicas Temporales

El desglose temporal mediante subgráficos perfila el **ciclo de vida de la siniestralidad urbana**:

**Distribución mensual — Patrón bimodal**

Julio marca el máximo anual (**789 accidentes**), seguido de cerca por junio y octubre. Agosto impone una caída drástica del **37,5%** (493 accidentes), reflejo directo del éxodo vacacional y la reducción de la movilidad pendular.

**Distribución semanal — Fuerte asimetría laborable-festivo**

Los laborables concentran el **83,5%** de los incidentes (miércoles a la cabeza con 1.319). El domingo cae a 655 casos. Este desplome no solo obedece a la reducción del volumen de tráfico, sino a un cambio en la **naturaleza del desplazamiento** (ocio frente a obligación) que altera factores cinemáticos.

**Distribución horaria — Curva de doble pico**

Alineada con los picos de estrés viario, las **14h** (540 casos) y las **17h** (534 casos) marcan las horas críticas, correspondientes a las transiciones laborales, frente al valle nocturno (1h–5h) con valores marginales.

## 6. Relación accidentes - condiciones meteorológicas

Exploramos si existe correlación visual entre las condiciones meteorológicas (temperatura, humedad, precipitación) y la gravedad de los accidentes.

In [19]:
# Scatter: Temperatura vs Humedad, coloreado por gravedad
fig_meteo = px.scatter(
    df_geo,
    x="TM",
    y="HRM",
    color="gravedad",
    color_discrete_map=SEVERITY_COLORS,
    opacity=0.5,
    hover_data=["nom_districte", "nom_barri", "PPT"],
    labels={"TM": "Temperatura media (°C)", "HRM": "Humedad relativa media (%)"},
    title="Accidentes según condiciones meteorológicas",
    category_orders={"gravedad": ["Mortal", "Grave", "Leve", "Sin víctimas"]},
)
fig_meteo.update_layout(height=500)
fig_meteo.show()

### Análisis: Visualización 5 — Condiciones Meteorológicas vs Gravedad

El análisis de dispersión cruzando temperatura, humedad y gravedad arroja un **hallazgo contraintuitivo pero causalmente robusto**: la precipitación media en días con accidentes graves/mortales (**1,063 mm**) es inferior a la de accidentes leves (**1,885 mm**).

Este fenómeno se explica por la **compensación de riesgo**: la lluvia evidente induce comportamientos defensivos (reducción de velocidad), mitigando la energía cinética del impacto. Por el contrario, el **clima seco e idóneo** (donde ocurren el 74,2% de los siniestros) facilita mayores velocidades y excesos de confianza.

> **Conclusión:** La temperatura se confirma como una variable de contexto estacional sin impacto directo sobre la mortalidad. El riesgo de mortalidad se dispara en días de buen tiempo, donde la percepción de seguridad relaja el comportamiento del conductor e incrementa la velocidad.

## 7. Análisis descriptivo por distrito

Comparativa de accidentes, víctimas y condiciones meteorológicas medias por distrito.

In [20]:
# Accidentes por distrito y gravedad (stacked bar)
severity_by_district = (
    df_geo.groupby(["nom_districte", "gravedad"])
    .agg(count=("numero_expedient", "count"))
    .reset_index()
)

fig_district = px.bar(
    severity_by_district,
    x="nom_districte",
    y="count",
    color="gravedad",
    color_discrete_map=SEVERITY_COLORS,
    barmode="stack",
    category_orders={"gravedad": ["Mortal", "Grave", "Leve", "Sin víctimas"]},
    labels={"nom_districte": "Distrito", "count": "Número de registros"},
    title="Distribución de gravedad por distrito",
)
fig_district.update_layout(height=500, xaxis_tickangle=-45)
fig_district.show()

## 8. Análisis por tipo de vehículo y perfil demográfico

Distribución de accidentes por tipo de vehículo implicado y perfil (sexo, edad) de las personas involucradas.

In [21]:
fig_vehicle = make_subplots(
    rows=1, cols=2,
    subplot_titles=("Accidentes por tipo de vehículo", "Distribución de edad por gravedad"),
    column_widths=[0.45, 0.55],
)

# Top 10 tipos de vehículo
vehicle_counts = df_geo["desc_tipus_vehicle_implicat"].value_counts().head(10)
fig_vehicle.add_trace(
    go.Bar(
        y=vehicle_counts.index[::-1],
        x=vehicle_counts.values[::-1],
        orientation="h",
        marker_color="#9b59b6",
        name="Vehículos",
    ),
    row=1, col=1,
)

# Distribución de edad por gravedad (box plot)
for sev in ["Mortal", "Grave", "Leve", "Sin víctimas"]:
    subset = df_geo[df_geo["gravedad"] == sev]
    if len(subset) > 0:
        fig_vehicle.add_trace(
            go.Box(
                y=subset["edat"],
                name=sev,
                marker_color=SEVERITY_COLORS[sev],
            ),
            row=1, col=2,
        )

fig_vehicle.update_layout(height=500, showlegend=False, title_text="Perfil de accidentes")
fig_vehicle.update_yaxes(title_text="Edad", row=1, col=2)
fig_vehicle.show()

### Análisis: Visualizaciones 6 y 7 — El Perfil del Riesgo (Vehículo y Demografía)

La caracterización de la siniestralidad se completa analizando los agentes implicados:

**El factor vehículo**

El **turismo (35,1%)** lidera el volumen, pero la **motocicleta (29,3%)** exhibe una preocupante sobrerepresentación. Pese a ser una fracción menor del parque móvil de Barcelona, su altísima participación confirma su profunda vulnerabilidad cinemática. La micromovilidad (bicicletas y VMP) suma más de 1.500 casos, exigiendo nuevas estrategias de pacificación.

**El factor humano**

Los boxplots de edad muestran homogeneidad entre categorías (la edad media orbita los **42–46 años** en todos los niveles de gravedad). Sin embargo, la **brecha de género** es determinante:

| Categoría | % Hombres |
|-----------|:---------:|
| Heridos graves | 71,2% |
| Fallecidos | 77,3% |

> **Implicación:** Las acciones de reeducación vial deben interpelar directamente a conductores masculinos de mediana edad, que constituyen el perfil de mayor riesgo de traumatismo severo en el ecosistema urbano de Barcelona.

## 9. Mapa animado temporal (Plotly)

Mapa interactivo con animación mes a mes que muestra cómo se distribuyen los accidentes geográficamente a lo largo del año. Cada punto representa un accidente, coloreado por gravedad.

In [22]:
# Preparar datos para animación mensual
# Usamos T_ACCIDENTS para evitar duplicados por persona
df_anim = df_accidents.dropna(subset=["latitud_wgs84", "longitud_wgs84"]).copy()
df_anim["gravedad"] = df_anim.apply(classify_severity, axis=1)

# Mapear mes numérico a nombre
MONTH_MAP = dict(zip(range(1, 13), MONTH_ORDER))
df_anim["nom_mes_ordered"] = df_anim["mes_any"].map(MONTH_MAP)

fig_anim = px.scatter_map(
    df_anim.sort_values("mes_any"),
    lat="latitud_wgs84",
    lon="longitud_wgs84",
    color="gravedad",
    color_discrete_map=SEVERITY_COLORS,
    animation_frame="nom_mes_ordered",
    hover_name="nom_barri",
    hover_data={"nom_districte": True, "nom_carrer": True, "numero_victimes": True,
                "latitud_wgs84": False, "longitud_wgs84": False},
    category_orders={
        "gravedad": ["Mortal", "Grave", "Leve", "Sin víctimas"],
        "nom_mes_ordered": [MONTH_MAP[i] for i in sorted(df_anim["mes_any"].unique())],
    },
    zoom=12,
    center={"lat": 41.392, "lon": 2.164},
    title="Evolución mensual de accidentes en Barcelona",
)
fig_anim.update_layout(map_style="carto-positron", height=650, margin=dict(l=0, r=0, t=40, b=0))
fig_anim.show()

## 10. Heatmap hora-día: cuándo ocurren más accidentes

Mapa de calor que cruza hora del día con día de la semana para identificar los momentos de mayor riesgo.

In [23]:
# Pivot table hora x día (usando T_ACCIDENTS)
heatmap_data = (
    df_accidents.groupby(["descripcio_dia_setmana", "hora_dia"])["numero_expedient"]
    .nunique()
    .reset_index()
    .pivot(index="descripcio_dia_setmana", columns="hora_dia", values="numero_expedient")
    .reindex(DAY_ORDER)
    .fillna(0)
)

fig_hm = px.imshow(
    heatmap_data,
    labels=dict(x="Hora del día", y="Día de la semana", color="Accidentes"),
    color_continuous_scale="YlOrRd",
    aspect="auto",
    title="Heatmap: accidentes por hora y día de la semana",
)
fig_hm.update_layout(height=400)
fig_hm.show()

### Análisis: Visualización 4 — Mapa de Calor Bidimensional (Hora × Día)

Este cruce matricial ofrece el **insight más directo para la operativa policial**. Las zonas de mayor intensidad (celdas rojas) revelan que:

- La franja de **13h a 18h de lunes a viernes** es el núcleo duro del riesgo estructural.
- El **viernes de 17h a 19h** representa el momento más crítico de la semana (operación salida de fin de semana).
- Las **madrugadas de fin de semana** (0h–4h) proyectan una señal débil en volumen, pero estadísticamente significativa por su asociación con movilidad de ocio y mayor severidad de las lesiones.

> **Directriz operativa:** El riesgo obedece a un patrón cronológico estricto. La franja de tarde (13h–19h) en laborables debe guiar la ubicación dinámica de dotaciones de emergencia y control de tráfico.

## 11. Exportar mapa interactivo a HTML

Guardamos el mapa Folium como archivo HTML para que pueda abrirse directamente en cualquier navegador sin necesidad de Jupyter.

In [24]:
output_path = "bcn_accidents_interactive_map.html"
m.save(output_path)
print(f"Mapa interactivo guardado en: {output_path}")
print("Ábrelo en un navegador para explorar todas las capas.")

Mapa interactivo guardado en: bcn_accidents_interactive_map.html
Ábrelo en un navegador para explorar todas las capas.


---

## Conclusiones Accionables del Modelado Descriptivo

La explotación de la `T_UNIFIED` a través de este pipeline arroja **cinco directrices clave** para la política pública de movilidad de Barcelona:

### 1. Focalización territorial
El **24,9%** del problema reside en el Eixample. Específicamente, el barrio de la **Dreta de l'Eixample** (8,7% del total de la ciudad) requiere intervenciones urgentes de rediseño viario.

### 2. Asignación de recursos predecible
El riesgo obedece a un **patrón cronológico estricto**. La franja de tarde (13h–19h) en laborables, con un pico de alerta los viernes por la tarde, debe guiar la ubicación dinámica de dotaciones de emergencia y control.

### 3. El falso confort climático
Las campañas preventivas **no deben restringirse a alertas por lluvia**. El riesgo de mortalidad se dispara en días de buen tiempo, donde la percepción de seguridad relaja el comportamiento del conductor e incrementa la velocidad.

### 4. La motocicleta como vector crítico
Con una participación que roza el **30%**, los usuarios de vehículos de dos ruedas motorizados asumen el mayor riesgo de traumatismo severo del ecosistema urbano.

### 5. Segmentación poblacional
El retrato del riesgo máximo (mortal o grave) tiene un **sesgo de género innegable**. Las acciones de reeducación vial deben interpelar directamente a conductores masculinos de mediana edad.